# 05 AI-policy features and master join

Turns the bill level NCSL table (`ncsl_ai_legislation/data/ncsl_ai_db_final.csv`, one row per bill,
built by `ncsl_ai_legislation/scripts/`) into **state-level policy features**, then joins them to the
*county table from notebook 02*.

**Why state level.** A county inherits its state's legislative record, so every policy feature is constant within a state. That's why notebook 05
validates on states that held out  & compares policy against a 'geography only' control.

Four dimensions of a state's AI-legislative record (2018–2025):

| Dimension | Features | Question it answers |
|---|---|---|
| Attention and follow through | `pol_bills_log`, `pol_enacted_log`, `pol_pass_rate` | How much AI law does the state consider, and how much passes? |
| Timing | `pol_first_year` | Was the state an early mover? |
| Content | `pol_share_regulatory`, `pol_share_public`, `pol_share_rights` | Do its bills regulate private companies, government use, or rights and safety? |
| Infrastructure bills | `pol_infra_bills`, `pol_infra_enacted` | Does it legislate the parts of AI that touch siting (energy, taxes, data centers)? |

**Outputs** `processed/stage3_model_table/state_policy_features.csv`, `model_table.csv`, `feature_dictionary.csv`.

In [ ]:
#!pip install geopandas     #uncomment if geopandas is not installed

import zipfile
import geopandas as gpd
import numpy as np
import pandas as pd

from pipeline_config import CLUSTER_FEATURES, DIVISIONS, GEO, INFRA, POLICY, ROOT, STAGE2, STAGE3

NCSL = ROOT / "data" / "ncsl_ai" / "output"                     # NCSL pipeline outputs
CENSUS = ROOT / "data" / "census" / "raw_data"                 # downloaded on first run
PRIOR = 5  # adding 5 imaginary bills to every state, see below

`PRIOR`: Some states introduce very few bills, South Dakota has only 2 regulatory AI bills. This causes the calculation of “share of bills that are regulatory” to have values like 0%, 50%, or 100%. This adds 5 bills to every state so the computations are smoother. Shares &
pass rate are shrunk toward the national value so states passing many bills won't be affected by 5 imaginary ones. See `shrunk_share` in the 'Aggregate to states' cell below.

## Bill level flags
Content groups built from category flags. Each bill already has dozens of individual **0/1** category flags like `category_private_sector`, `category_notification`, etc. *This collapses them into three broader buckets, for ex: is this bill about regulating private industry, government/public use, or rights and safety?* For each bucket it takes the **max** across the relevant category columns row wise, so if a bill is flagged **1** in any of the categories belonging to a group, the whole group gets **1**.

In [ ]:
bills = pd.read_csv(NCSL / "ncsl_ai_db_final.csv")
text = pd.read_csv(NCSL / "ncsl_ai_db_clean.csv", usecols=["bill_id", "summary"])
# 2 fallback ids were renamed in 'clean file' (Oct 21) after the final file was written, match them back.
text["bill_id"] = text.bill_id.replace({"NJ2023A5969": "New Jersey_2021_A5969", "NC2021H259": "North Carolina_2023_H259"})
bills = bills.merge(text, on="bill_id", how="left", validate="1:1", indicator=True)

assert bills.bill_id.is_unique and len(bills) == 1907 and (bills._merge == "both").all(), "a bill has no summary"
bills = bills.drop(columns="_merge")

# Same groups as features.ipynb, except that provenance is
# counted in rights and safety (features.ipynb looks it up as "category_category_provenance").
GROUPS = {"regulatory": ["private_sector", "notification", "private_right_of_action", "impact_assessment",
                         "oversight_governance", "responsible_use"],
          "public": ["government_use", "judicial_use", "appropriations", "audit"],
          "rights": ["criminal_use", "child_pornography", "elections", "personhood", "provenance"]}

for name, cats in GROUPS.items():
    bills[f"grp_{name}"] = bills[[f"category_{c}" for c in cats]].max(axis=1)

words = (bills.title.fillna("") + " " + bills.summary.fillna("")).str.lower()
bills["is_passed"] = bills.status_type.eq("Passed")
bills["is_decided"] = bills.status_type.isin(["Passed", "Failed", "Vetoed"])
bills["mentions_data_center"] = words.str.contains("data center")
bills["is_infra"] = bills.category_energy.eq(1) | bills.category_taxes.eq(1) | bills.mentions_data_center

#Check that counts match original data
print(f"{len(bills):,} bills | passed {bills.is_passed.sum()} | decided {bills.is_decided.sum()} | "
      f"energy {bills.category_energy.sum()} | taxes {bills.category_taxes.sum()} | "
      f"mention data centers {bills.mentions_data_center.sum()} | infrastructure-related {bills.is_infra.sum()}")

1,907 bills | passed 325 | decided 1050 | energy 35 | taxes 20 | mention data centers 16 | infrastructure-related 48


## Aggregate bills to states
Groups bills by state and builds one row per state with raw counts (bills introduced, enacted, decided, first session year, infrastructure related bills) plus whether any infra bill was passed. Applies the shrinkage formula to each state’s enacted/decided legislation ratio. It is 'pulled' toward the **31% national rate**, & weighted by how many decided bills that state actually has. Log transforms the raw infrastructure bill count. 

In [21]:
g = bills.groupby("state")
n = g.size()

def shrunk_share(count, total, national):
    """(count + PRIOR * national) / (total + PRIOR) so 2 bill state can't reach 100% share."""
    return (count + PRIOR * national) / (total + PRIOR)

nat_pass = bills.is_passed.sum() / bills.is_decided.sum()
state = pd.DataFrame({
    "pol_bills": n,
    "pol_enacted": g.is_passed.sum(),
    "pol_decided": g.is_decided.sum(),
    "pol_first_year": g.session_year.min(),
    "pol_infra_bills_raw": g.is_infra.sum(),
    "pol_infra_enacted": g.apply(lambda b: int((b.is_infra & b.is_passed).any()), include_groups=False)})

state["pol_bills_log"] = np.log1p(state.pol_bills)
state["pol_enacted_log"] = np.log1p(state.pol_enacted)
state["pol_pass_rate"] = shrunk_share(state.pol_enacted, state.pol_decided, nat_pass)

for col, flag in [("pol_share_regulatory", "grp_regulatory"), ("pol_share_public", "grp_public"),
                  ("pol_share_rights", "grp_rights")]:
    state[col] = shrunk_share(g[flag].sum(), n, bills[flag].mean())

state["pol_infra_bills"] = np.log1p(state.pol_infra_bills_raw)
state = state.reset_index().rename(columns={"state": "state_abbr"})

assert set(POLICY) <= set(state.columns)
print(f"national pass rate among decided bills: {nat_pass:.3f}")
state.sort_values("pol_bills", ascending=False).head(8).round(3)

national pass rate among decided bills: 0.310


,state_abbr,pol_bills,pol_enacted,pol_decided,pol_first_year,pol_infra_bills_raw,pol_infra_enacted,pol_bills_log,pol_enacted_log,pol_pass_rate,pol_share_regulatory,pol_share_public,pol_share_rights,pol_infra_bills
35,NY,230,8,29,2019,3,0,5.442,2.197,0.281,0.716,0.301,0.247,1.386
4,CA,123,12,31,2019,2,0,4.820,2.565,0.376,0.503,0.365,0.141,1.099
15,IL,108,11,30,2019,0,0,4.691,2.485,0.359,0.516,0.201,0.115,0.000
32,NJ,99,6,16,2018,15,1,4.605,1.946,0.359,0.551,0.334,0.058,2.773
20,MA,96,4,21,2019,2,0,4.575,1.609,0.213,0.647,0.334,0.089,1.099
45,TX,96,26,96,2019,2,1,4.575,3.296,0.273,0.380,0.334,0.297,1.099
21,MD,89,23,89,2019,1,0,4.500,3.178,0.261,0.461,0.401,0.149,0.693
12,HI,81,7,56,2019,0,0,4.407,2.079,0.140,0.411,0.380,0.070,0.000


## Census division (the geography control) and county population (diagnostic only)

 Produces three small lookup tables (div, cpop, area) that get joined onto the county master table in the next cell. Only **division** feeds model. **Population** and **area** are not model features, they exist so notebook 06 can check whether prediction errors track with **demand** but none of our sources carries that variable.

In [22]:
SOURCES = {"co-est2024-alldata.csv": "https://www2.census.gov/programs-surveys/popest/datasets/2020-2024/counties/totals/co-est2024-alldata.csv",
           "cb_2024_us_county_500k.zip": "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_500k.zip"}

CENSUS.mkdir(parents=True, exist_ok=True)
for name, url in SOURCES.items():                      # first run only fetch two public Census files
    if not (CENSUS / name).exists():
        import urllib.request
        urllib.request.urlretrieve(url, CENSUS / name)

pop = pd.read_csv(CENSUS / "co-est2024-alldata.csv", encoding="latin-1", dtype={"STATE": str, "COUNTY": str})
div = (pop[pop.SUMLEV == 40][["STATE", "DIVISION"]].astype({"DIVISION": int})
       .rename(columns={"STATE": "state_fips"}))

div["division"] = div.DIVISION.map(dict(enumerate(DIVISIONS, start=1)))
cpop = pop[pop.SUMLEV == 50].assign(county_fips=lambda d: d.STATE + d.COUNTY)[["county_fips", "POPESTIMATE2024"]]
cpop = cpop.rename(columns={"POPESTIMATE2024": "population_2024"})

with zipfile.ZipFile(CENSUS / "cb_2024_us_county_500k.zip") as z:
    z.extractall(CENSUS / "cb_2024_us_county_500k")

shapes = gpd.read_file(CENSUS / "cb_2024_us_county_500k")
area = pd.DataFrame({"county_fips": shapes.GEOID, "land_sq_km": shapes.ALAND / 1e6})

## Join
Loads the county table produced earlier, drops 1 row that’s a malformed placeholder FIPS ("00000"). Then flags which counties have every clustering/infrastructure feature present (no missing values) w/ the `complete` mask.

In [25]:
master = pd.read_csv(STAGE2 / "master_county_clean.csv", dtype={"county_fips": str, "state_fips": str})
master = master[master.county_fips != "00000"]                       # malformed FIPS reference row
complete = master[CLUSTER_FEATURES + INFRA].notna().all(axis=1)
print(f"{len(master):,} counties in the master table, {complete.sum():,} with complete CERF features")

d = master[complete].copy()
d["has_data_center"] = (d.atlas_record_count > 0).astype(int)

d = d.merge(state[["state_abbr"] + POLICY + ["pol_bills", "pol_enacted", "pol_infra_bills_raw"]],
            on="state_abbr", how="left", validate="m:1", indicator="policy_join")
d = d.merge(div[["state_fips", "division"]], on="state_fips", how="left", validate="m:1")

for name, col in zip(DIVISIONS, GEO):
    d[col] = (d.division == name).astype(int)
d = d.merge(cpop, on="county_fips", how="left", validate="1:1").merge(area, on="county_fips", how="left", validate="1:1")
d["pop_density"] = d.population_2024 / d.land_sq_km

# Check that merges worked 
assert (d.policy_join == "both").all(), "a county's state has no NCSL record"
assert d.division.notna().all() and d[GEO].sum(axis=1).eq(1).all()
missing_pop = d[d.population_2024.isna()][["county_fips", "county_name", "state_abbr"]]
print(f"{len(d):,} counties | {d.has_data_center.sum()} with a data center ({d.has_data_center.mean():.1%}) | "
      f"{d.state_abbr.nunique()} states | {d.atlas_record_count.sum():,} atlas facilities")

print("counties without a 2024 population estimate:", missing_pop.to_dict("records"))

3,153 counties in the master table, 3,099 with complete CERF features
3,099 counties | 244 with a data center (7.9%) | 48 states | 1,462 atlas facilities
counties without a 2024 population estimate: []


In [26]:
keep = (["county_fips", "state_fips", "state_abbr", "county_name", "centroid_longitude", "centroid_latitude",
         "division", "atlas_record_count", "has_data_center"] + list(dict.fromkeys(CLUSTER_FEATURES + INFRA))
        + GEO + POLICY + ["pol_bills", "pol_enacted", "pol_infra_bills_raw", "population_2024", "land_sq_km", "pop_density"])

d[keep].to_csv(STAGE3 / "model_table.csv", index=False)
state.to_csv(STAGE3 / "state_policy_features.csv", index=False)

dictionary = pd.DataFrame([
    ("pol_bills_log", "Attention and follow-through", "ln(1 + AI bills introduced, 2018-2025 sessions)"),
    ("pol_enacted_log", "Attention and follow-through", "ln(1 + AI bills enacted or adopted)"),
    ("pol_pass_rate", "Attention and follow-through", f"Share of decided bills that passed, shrunk to the national {nat_pass:.2f} with a 5-bill prior"),
    ("pol_first_year", "Timing", "First session year with an AI bill (biennial sessions count from their first year)"),
    ("pol_share_regulatory", "Content", "Share of bills regulating private-sector AI (private sector, notification, impact assessment, oversight, responsible use, private right of action), shrunk"),
    ("pol_share_public", "Content", "Share of bills on government, judicial use, appropriations or audit, shrunk"),
    ("pol_share_rights", "Content", "Share of bills on criminal use, AI-generated child sexual abuse material, elections, personhood or provenance, shrunk"),
    ("pol_infra_bills", "Infrastructure bills", "ln(1 + bills tagged energy or taxes, or mentioning data centers)"),
    ("pol_infra_enacted", "Infrastructure bills", "1 if any such bill passed"),
], columns=["feature", "group", "definition"])

dictionary.to_csv(STAGE3 / "feature_dictionary.csv", index=False)
print(f"wrote model_table.csv ({len(d):,} x {len(keep)}), state_policy_features.csv ({len(state)} jurisdictions)")
d[POLICY].describe().T.round(3)

wrote model_table.csv (3,099 x 45), state_policy_features.csv (54 jurisdictions)


,count,mean,std,min,25%,50%,75%,max
pol_bills_log,3099.0,3.377,0.842,1.099,2.833,3.219,3.829,5.442
pol_enacted_log,3099.0,1.900,0.734,0.000,1.099,1.792,2.485,3.296
pol_pass_rate,3099.0,0.343,0.146,0.122,0.237,0.283,0.434,0.713
pol_first_year,3099.0,2020.232,1.621,2018.000,2019.000,2019.000,2021.000,2024.000
pol_share_regulatory,3099.0,0.397,0.101,0.214,0.346,0.380,0.471,0.716
pol_share_public,3099.0,0.367,0.114,0.124,0.301,0.338,0.448,0.643
pol_share_rights,3099.0,0.237,0.098,0.058,0.154,0.220,0.297,0.454
pol_infra_bills,3099.0,0.389,0.540,0.000,0.000,0.000,0.693,2.773
pol_infra_enacted,3099.0,0.176,0.381,0.000,0.000,0.000,0.000,1.000
